In [2]:
import numpy as np
import sympy as smp

In [3]:
def verify_convexity(f, variables):
    print("\n=== Convexity Verification ===")
    hessian_sym = smp.hessian(f, variables)
    print("Hessian Matrix:")
    display(hessian_sym)

    try:
        hessian_np = np.array(hessian_sym).astype(np.float64)
    except TypeError:
        print("Hessian contains variables. Convexity depends on the specific coordinate.")
        return True

    eigenvalues = np.linalg.eigvals(hessian_np)
    print(f"Eigenvalues: {eigenvalues}")

    if np.all(eigenvalues >= 0):
        print("\tThe function IS convex.")
        return True
    else:
        print("\tThe function IS NOT convex.")
        return False

In [4]:
def set_gradient(variables, f):
    gradient_expressions = []
    print(f"Derivatives for all variables: ")

    for v in variables:
        df = smp.diff(f, v)
        display(df)
        gradient_expressions.append(df)

    gradient_functions = []
    for expr in gradient_expressions:
        gradient_functions.append(smp.lambdify(variables, expr, 'numpy'))

    return gradient_functions

In [5]:
def compute_gradient(parameters, gradient_functions):
    return np.array([x(*parameters) for x in gradient_functions])

In [6]:
def compute_alpha(f, variables, parameters, gradient):
    alpha = smp.symbols('alpha', real = True)

    alpha_subs_dict = {}
    for i in range(len(variables)):
        alpha_subs_dict[variables[i]] = parameters[i] - alpha * gradient[i]

    f_alpha = f.subs(alpha_subs_dict)
    df_alpha = smp.diff(f_alpha, alpha)

    solution = smp.solve(df_alpha, alpha)
    for s in solution:
        if s.is_real:
            return np.float64(s)

    return None

In [7]:
def fastest_grad_descent(variables, f, tol, static_alpha = False):

    if not verify_convexity(f, variables):
        return None

    print("\n=== Start of training ===")
    x0 = np.ones(len(variables), dtype = np.float64)
    x = np.copy(x0)

    f_numpy = smp.lambdify([*variables], f, 'numpy')
    grad_functions = set_gradient(variables, f)

    count = 0
    while True:
        count += 1

        grad = compute_gradient(x0, grad_functions)

        if np.isnan(grad).any() or np.isinf(grad).any():
            print("ERROR: Gradient resulted in NaN or Infinity. The function is likely not continuous here.")
            return None

        if np.linalg.norm(grad)  < tol:
            break

        alpha = 0.1 if static_alpha else compute_alpha(f, variables, x0, grad)

        x -= alpha * grad
        if (count % 10) == 0:
            print(f"{count} iteration,\tGradient = {grad}\tAlpha = {alpha} \t x = {x}\tf(x) = {f_numpy(*x)}")
        x0 = np.copy(x)

    print(f"Number of iterations: {count}")
    print("\n===Gradient Descent successfully found local minimum===")
    print(f"\tat x = {x0}, f(x) = {f_numpy(*x0)}")
    print("\tfor function:")
    display(f)

    return x0

In [12]:
smp.init_printing()
x1, x2, x3 = smp.symbols('x1 x2 x3', real=True, dtype = np.float64)
variables = [x1, x2, x3]

f = 3 * x1 * x1 + 4 * x2 * x2 + 5 * x3 * x3 + 2 * x1 * x2 - x1 * x3 - 2 * x2 * x3 - 3 * x3

In [13]:
print("====== Gradient Descent with static learning rate: ======")
local_minimum = fastest_grad_descent(variables, f, 1e-9, static_alpha=True)

====== Gradient Descent with static learning rate: ======

=== Convexity Verification ===
Hessian Matrix:


⎡6   2   -1⎤
⎢          ⎥
⎢2   8   -2⎥
⎢          ⎥
⎣-1  -2  10⎦

Eigenvalues: [11.88089912  4.75418976  7.36491112]
	The function IS convex.

=== Start of training ===
Derivatives for all variables: 


10 iteration,	Gradient = [ 0.00330094 -0.00209279 -0.00014661]	Alpha = 0.1 	 x = [0.02920826 0.0718815  0.31728828]	f(x) = -0.4759610958371479
20 iteration,	Gradient = [ 5.16385453e-06 -3.36597511e-06 -2.98889638e-07]	Alpha = 0.1 	 x = [0.02884672 0.07211501 0.31730766]	f(x) = -0.4759615384604362
30 iteration,	Gradient = [ 8.14884071e-09 -5.31184219e-09 -4.71775063e-10]	Alpha = 0.1 	 x = [0.02884615 0.07211538 0.31730769]	f(x) = -0.47596153846153844
Number of iterations: 34

===Gradient Descent successfully found local minimum===
	at x = [0.02884615 0.07211538 0.31730769], f(x) = -0.4759615384615385
	for function:


In [14]:
print("=== Gradient Descent with FASTEST learning rate: ===")
local_minimum = fastest_grad_descent(variables, f, 1e-9)

=== Gradient Descent with FASTEST learning rate: ===

=== Convexity Verification ===
Hessian Matrix:


⎡6   2   -1⎤
⎢          ⎥
⎢2   8   -2⎥
⎢          ⎥
⎣-1  -2  10⎦

Eigenvalues: [11.88089912  4.75418976  7.36491112]
	The function IS convex.

=== Start of training ===
Derivatives for all variables: 


10 iteration,	Gradient = [ 0.00012787 -0.00046132  0.00035997]	Alpha = 0.10450917571381572 	 x = [0.02887957 0.07210129 0.31729829]	f(x) = -0.47596153476678005
20 iteration,	Gradient = [ 1.54672752e-08 -5.58115073e-08  4.35462248e-08]	Alpha = 0.10450917583947994 	 x = [0.02884616 0.07211538 0.31730769]	f(x) = -0.47596153846153844
Number of iterations: 25

===Gradient Descent successfully found local minimum===
	at x = [0.02884615 0.07211538 0.31730769], f(x) = -0.4759615384615385
	for function:
